In [ ]:
# Cell 1: imports
import numpy as np
import matplotlib.pyplot as plt

import three_dsx as sx

In [ ]:
# Cell 2: build a simple synthetic 3D geometry

# Receiver lines along x, sources along y (very simple grid)
rx = np.repeat(np.arange(0, 2000, 200), 10)
ry = np.tile(np.arange(0, 1000, 100), 10)

sx = np.repeat(np.arange(0, 2000, 200), 10)
sy = np.tile(np.arange(0, 1000, 100), 10)

offsets = sx.compute_offsets(rx, ry, sx, sy)
azimuths = sx.compute_azimuths(rx, ry, sx, sy)

In [ ]:
# Cell 3: basic offset diagnostics

min_off, max_off = 0, 2500
avg_sep = 200
disc = 150

print("Multiplicity:", sx.multiplicity(offsets, min_off, max_off))
print("Offset uniformity:", sx.offset_uniformity(offsets, min_off, max_off, avg_sep))
print("Discrimination multiplicity:", sx.discrimination_multiplicity(offsets, disc))
print("Effective fold:", sx.effective_fold(offsets, min_off, max_off, avg_sep))
print("Delta-offset RMS:", sx.delta_offset_rms(offsets))
print("Offset range:", sx.offset_range(offsets))
print("Largest gap:", sx.largest_gap(offsets))

In [ ]:
# Cell 4: offset and azimuth histograms

plt.figure(figsize=(12,4))

plt.subplot(1,2,1)
plt.hist(offsets, bins=20)
plt.title("Offset Histogram")
plt.xlabel("Offset (m)")

plt.subplot(1,2,2)
plt.hist(np.degrees(azimuths) % 360, bins=36)
plt.title("Azimuth Histogram")
plt.xlabel("Azimuth (deg)")

plt.tight_layout()
plt.show()

In [ ]:
# Cell 5: spider plot (offset vs azimuth in polar coordinates)

plt.figure(figsize=(6,6))
ax = plt.subplot(111, projection='polar')
ax.scatter(azimuths, offsets, s=5)
ax.set_title("Spider Plot")
plt.show()

In [ ]:
# Cell 6: multiple suppression (no arrays)

fmin, fmax = 5, 60      # Hz
VM, VP = 2000, 2500     # m/s
t0 = 1.0                # s

R_mult = sx.multiple_suppression(offsets, fmin, fmax, VM, VP, t0)
print("Multiple suppression (no arrays):", R_mult)

In [ ]:
# Cell 7: stack response over wavelength range

lam_min, lam_max = 50, 500  # m

R_stack = sx.stack_response(offsets, lam_min, lam_max)
print("Stack response (no arrays):", R_stack)

In [ ]:
# Cell 8: simple linear source array and its response

# 5 elements, 10 m spacing along x
elements = np.column_stack([np.arange(-20, 30, 10), np.zeros(5)])
weights = np.ones(5)

lam = 100.0          # m
theta = np.radians(45)  # 45° noise azimuth

R_arr = sx.array_response(elements, weights, lam, theta)
print("Array response magnitude:", np.abs(R_arr))
print("Array response phase (deg):", np.degrees(np.angle(R_arr)))

In [ ]:
# Cell 9: wavelength–azimuth response map

lams = np.linspace(50, 400, 40)
thetas = np.linspace(0, 2*np.pi, 72)

Rmap = np.zeros((len(lams), len(thetas)))

for i, L in enumerate(lams):
    for j, th in enumerate(thetas):
        Rmap[i,j] = np.abs(sx.array_response(elements, weights, L, th))

plt.figure(figsize=(8,5))
plt.imshow(Rmap, aspect='auto', origin='lower',
           extent=[0, 360, lams[0], lams[-1]], cmap='viridis')
plt.colorbar(label="|R|")
plt.xlabel("Azimuth (deg)")
plt.ylabel("Wavelength (m)")
plt.title("Array Response |R(λ, θ)|")
plt.show()

In [ ]:
# Cell 10: DMO spray for a given dip and azimuth

dip_deg = 20
az_deg = 45
V = 2000
t0 = 1.0

shift_x, shift_y = sx.dmo_point_movement(offsets, dip_deg, az_deg, V, t0)

plt.figure(figsize=(6,4))
plt.plot(offsets, shift_x, 'o-', label="Inline shift")
plt.plot(offsets, shift_y, 's-', label="Crossline shift")
plt.xlabel("Offset (m)")
plt.ylabel("DMO Shift (m)")
plt.title("DMO Spray")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Cell 11: DMO binning and fold map

# Use midpoints as CMP positions
cmp_x = 0.5 * (rx + sx)
cmp_y = 0.5 * (ry + sy)

bin_size = 100.0

bins = sx.dmo_binning(cmp_x, cmp_y, offsets, dip_deg, az_deg, V, t0, bin_size)

xs = [i*bin_size for (i,j) in bins.keys()]
ys = [j*bin_size for (i,j) in bins.keys()]
vals = list(bins.values())

plt.figure(figsize=(6,6))
sc = plt.scatter(xs, ys, c=vals, s=200, cmap='viridis')
plt.colorbar(sc, label="DMO Fold")
plt.title("DMO Fold Map")
plt.xlabel("X (m)")
plt.ylabel("Y (m)")
plt.axis("equal")
plt.show()